# LangGraph Voice Translation Agent (MLX Native + Local LLM)

This notebook demonstrates a professional voice translation pipeline using:
1. **mlx-audio 0.4.0**: Native ASR (Qwen3-ASR) and TTS (Qwen3-TTS) on Apple Silicon.
2. **LangChain + Local LLM**: Qwen-based translation via an OpenAI-compatible API.

## 1. Setup and Dependencies

Ensure your environment is set up with the required libraries:
```bash
uv add langgraph mlx-audio==0.2.9 huggingface_hub langchain-openai transformers torch librosa
```

### Local LLM Requirement
This agent assumes a local OpenAI-compatible API is running (e.g., via vLLM, Ollama, or LM Studio) at `http://localhost:8000/v1`.

In [1]:
import os
import sys
from typing import TypedDict, Annotated, Dict
import asyncio
from langgraph.graph import StateGraph, END

# Add src to path if needed
sys.path.append(os.path.abspath("."))
from pipeline import VoiceTranslationPipeline

## 2. Define Agent State

In [2]:
class AgentState(TypedDict):
    audio_path: str
    src_lang: str
    tgt_lang: str
    out_audio_path: str
    original_text: str
    translated_text: str
    metrics: Dict[str, float]

## 3. Implement Workflow Nodes

In [3]:
pipeline = VoiceTranslationPipeline()

async def stt_node(state: AgentState):
    import time
    print("--- STT NODE (MLX NATIVE) ---")
    start = time.time()
    text = pipeline.transcribe_audio(state["audio_path"])
    return {
        "original_text": text,
        "metrics": {"stt_time": time.time() - start}
    }

async def mt_node(state: AgentState):
    import time
    print("--- MT NODE (LOCAL LLM) ---")
    start = time.time()
    translated = pipeline.translate_text(
        state["original_text"], 
        state["src_lang"], 
        state["tgt_lang"]
    )
    metrics = state.get("metrics", {})
    metrics["mt_time"] = time.time() - start
    return {
        "translated_text": translated,
        "metrics": metrics
    }

async def tts_node(state: AgentState):
    import time
    print("--- TTS NODE (MLX NATIVE) ---")
    start = time.time()
    await pipeline.text_to_speech(
        state["translated_text"], 
        state["tgt_lang"], 
        state["out_audio_path"]
    )
    metrics = state.get("metrics", {})
    metrics["tts_time"] = time.time() - start
    return {
        "metrics": metrics
    }

async def cleanup_node(state: AgentState):
    print("--- CLEANUP NODE ---")
    pipeline.clear_memory()
    return state

Initializing Native MLX-Audio Pipeline
Models directory: /Users/nitinkumar/Work/demo/models
Using local model: /Users/nitinkumar/Work/demo/models/asr
Using local model: /Users/nitinkumar/Work/demo/models/tts


You are using a model of type qwen3_tts to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
The tokenizer you are loading from '/Users/nitinkumar/Work/demo/models/tts' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  Initialized encoder codebooks
Loaded speech tokenizer from /Users/nitinkumar/Work/demo/models/tts/speech_tokenizer
Initializing Local Translation LLM at: http://127.0.0.1:1234/v1 with model: qwen3.5-0.8b


## 4. Assemble and Run Graph

In [4]:
workflow = StateGraph(AgentState)
workflow.add_node("stt", stt_node)
workflow.add_node("mt", mt_node)
workflow.add_node("tts", tts_node)
workflow.add_node("cleanup", cleanup_node)

workflow.set_entry_point("stt")
workflow.add_edge("stt", "mt")
workflow.add_edge("mt", "tts")
workflow.add_edge("tts", "cleanup")
workflow.add_edge("cleanup", END)

app = workflow.compile()

In [5]:
async def run_agent():
    audio_input = "../inputs/input.wav"
    output_audio = "../outputs/output.wav"
    
    initial_state = {
        "audio_path": audio_input,
        "src_lang": "en",
        "tgt_lang": "es",
        "out_audio_path": output_audio,
        "original_text": "",
        "translated_text": "",
        "metrics": {}
    }

    print("Invoking Voice Translation Agent...")
    final_state = await app.ainvoke(initial_state)
    
    print(f"\nResulting Translation: {final_state['translated_text']}")
    print("Metrics:", final_state['metrics'])

await run_agent()

Invoking Voice Translation Agent...
--- STT NODE (MLX NATIVE) ---
Transcribing audio: ../inputs/input.wav
--- MT NODE (LOCAL LLM) ---
Translating via Local LLM: en -> es
--- TTS NODE (MLX NATIVE) ---
Generating TTS for language: es
Text: Mejoré el foco en la pantalla mientras Jasperit Boomer proclamó cuatrocientas y cinco décimas, destruyendo Nueva Zelanda por 159 y cerrando el tercer título de India con 96 carreras. El tensión alcanzó su punto máximo con los rápidos cotes de James Nisham, pero el surgo tardío de Shivam Dubey de 26 carreras en ocho bales clausuró la partida analíticamente. La intención del truco de Boomer y el control de la tasa de hito hicieron que este fuera un masterclass en presión de bowling por muerte.
Voice: af_heart
Speed: 1.0x
Language: en


✅ Audio successfully generated and saving as: ../outputs/output.wav/audio_000.wav
Duration:              00:01:36.000
Samples/sec:           86503.2
Prompt:                1200 tokens, 45.1 tokens-per-sec
Audio:                 2304000 samples, 86503.2 samples-per-sec
Real-time factor:      3.60x
Processing time:       26.63s
Peak memory usage:     16.12GB
--- CLEANUP NODE ---
Clearing models from memory...
Memory cleanup complete.

Resulting Translation: Mejoré el foco en la pantalla mientras Jasperit Boomer proclamó cuatrocientas y cinco décimas, destruyendo Nueva Zelanda por 159 y cerrando el tercer título de India con 96 carreras. El tensión alcanzó su punto máximo con los rápidos cotes de James Nisham, pero el surgo tardío de Shivam Dubey de 26 carreras en ocho bales clausuró la partida analíticamente. La intención del truco de Boomer y el control de la tasa de hito hicieron que este fuera un masterclass en presión de bowling por muerte.
Metrics: {'stt_time': 2.9725730419158936, '